In [2]:
from science_jubilee.Machine import Machine
from science_jubilee.tools.Camera import Camera
from xarm.wrapper import XArmAPI

from plate_handler import PlateHandler

from pathlib import Path
from datetime import datetime
import time
import cv2

def capture_image(
    camera_index: int = 0,
    output_directory: str = "images/Jubilee",
) -> Path:
    """
    Capture one image from the Jubilee camera.

    Parameters
    ----------
    camera_index:
        OpenCV camera index.

    output_directory:
        Directory in which captured images are saved.

    Returns
    -------
    Path
        Path of the saved image.
    """
    output_path = Path(output_directory)
    output_path.mkdir(parents=True, exist_ok=True)

    camera_capture = cv2.VideoCapture(camera_index)

    if not camera_capture.isOpened():
        raise RuntimeError(
            f"Could not open camera index {camera_index}."
        )

    try:
        # Discard several initial frames so exposure and white balance can
        # stabilize before the final image is captured.
        for _ in range(10):
            camera_capture.read()

        time.sleep(2)

        success, frame = camera_capture.read()

        if not success or frame is None:
            raise RuntimeError(
                f"Failed to capture an image from camera {camera_index}."
            )

        image_path = (
            output_path
            / f"jubilee_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        )

        if not cv2.imwrite(str(image_path), frame):
            raise RuntimeError(
                f"Failed to save image to {image_path}."
            )

        print(f"Image saved: {image_path}")
        return image_path

    finally:
        camera_capture.release()

SDK_VERSION: 1.18.5


# Initialize Jubilee

if stuck, run `sudo ifconfig en8 inet 192.168.2.100 netmask 255.255.255.0 up`

In [3]:
jubilee = Machine(address="192.168.2.2")

camera_tool = Camera(1, "Camera")
jubilee.load_tool(camera_tool)

0
response in connect:  [False, False, False, False]


home Jubilee if needed

In [ ]:
jubilee.home_all()

# Initialize xArm

if failed connection, in terminal run `sudo ifconfig en18 inet 192.168.1.100 netmask 255.255.255.0 up`

In [2]:
ARM_IP = '192.168.1.205'  # change this to xArm IP
arm = XArmAPI(ARM_IP, do_not_open=True)
arm.connect()
plate_handler = PlateHandler(arm)
plate_handler.initialize()

ROBOT_IP: 192.168.1.205, VERSION: v2.7.0, PROTOCOL: V1, DETAIL: 6,6,XI1304,AC1303,v2.7.0, TYPE1300: [1, 1]
change protocol identifier to 3
[clean_error], xArm is ready to move
[set_state], xArm is ready to move


# Register Positions

In [3]:
plate_handler.register_position("Shelf_1")

Initial yaw correction: 5.711° | Fine yaw correction: -1.432°
Coarse edge detection after 145.00 mm of positive-Y travel.
Plate edge precisely detected after 143.50 mm of positive-Y travel.
Moved 145.00 mm in negative Y from the detected edge to the plate center.


{'position_id': 'Shelf_1',
 'pose': array([ 4.30068726e+02,  3.69475159e+02,  2.35287003e+02,  1.79877292e+02,
        -3.43947000e-01,  8.54776250e+01]),
 'loaded_plate': None}

In [ ]:
plate_handler.register_position("Shelf_2")

In [4]:
plate_handler.register_position("Jubilee")

Initial yaw correction: -0.000° | Fine yaw correction: 0.716°
Coarse edge detection after 150.00 mm of positive-Y travel.
Plate edge precisely detected after 149.50 mm of positive-Y travel.
Moved 145.00 mm in negative Y from the detected edge to the plate center.


{'position_id': 'Jubilee',
 'pose': array([ 1.92622177e+02, -4.78248230e+02,  2.42055954e+02, -1.79695092e+02,
         1.73033000e-01, -1.03464260e+02]),
 'loaded_plate': None}

In [7]:
arm.set_tool_position(x=-100, speed=70, wait=True)

0

In [ ]:
plate_handler.register_position("")
plate_handler.move_plate("plate_1", "Shelf_1")

In [ ]:
plate_handler._align_yaw()

check registered plate_postions

In [ ]:
plate_handler.plate_positions

In [ ]:
plate_handler.register_plate(
    plate_id="plate_1",
    position_id="Jubilee",
)

In [ ]:
plate_handler._pick_up_plate(plate_handler.get_position("Jubilee")["pose"])

# Associate Plate with Registered Positions

register the plate positions
- plate_1 starts at Shelf_1
- plate_2 starts at Shelf_2
- Jubilee starts empty

In [ ]:
plate_handler.register_plate(
    plate_id="plate_1",
    position_id="Shelf_1",
)

plate_handler.register_plate(
    plate_id="plate_2",
    position_id="Shelf_2",
)

check plate positions

In [ ]:
plate_handler.get_status()

# Example Workflow: Pickup plate on shelf, take photo, put back

In [ ]:
# Record plate_1's current position before moving it
plate_1_original_position = plate_handler.get_plate_position("plate_1")

# Move plate_1 from its current position to Jubilee
plate_handler.move_plate(
    plate_id="plate_1",
    destination_position_id="Jubilee",
    move_speed=100,
)


# Move the camera to the capture position and take an image
jubilee.move_to(x=150, y=150, z=240)
capture_image()

# Move the camera away before the xArm moves again
jubilee.move_to(x=300, y=10, z=240)


# Return plate_1 to the position it originally came from
plate_handler.move_plate(
    plate_id="plate_1",
    destination_position_id=plate_1_original_position,
    move_speed=100,
)


# Record plate_2's current position before moving it
plate_2_original_position = plate_handler.get_plate_position("plate_2")

# Move plate_2 from its current position to Jubilee
plate_handler.move_plate(
    plate_id="plate_2",
    destination_position_id="Jubilee",
    move_speed=100,
)


# Move the camera to the capture position and take an image
jubilee.move_to(x=150, y=150, z=240)
capture_image()

# Move the camera away before the xArm moves again
jubilee.move_to(x=300, y=10, z=240)

# Return plate_2 to the position it originally came from
plate_handler.move_plate(
    plate_id="plate_2",
    destination_position_id=plate_2_original_position,
    move_speed=100,
)


# Park the Jubilee camera tool
jubilee.park_tool()